## README
You need 3 files to run this code:
- AnnGeno.ag file for genotypes and phenotypes. If you don't have this file run code xx
- burdens.zarr for gene burden scores. If you don't have this file run compute_burdens.ipynb
- PRS.parquet for PRS scores per phenotype. If the PRS columns have different column names, you will need a PRS_id to phenotype mapping file
- associations.parquet file with the all the gene-trait associations to test


In [ ]:
import sys
import yaml
import zarr
import pandas as pd
import numpy as np
from tqdm import tqdm
from anngeno import AnnGeno
import statsmodels.api as sm
import matplotlib.pyplot as plt
from plotnine import *


In [ ]:
# Get covariate corrected phenotypes
def cov_prs_correction(all_df, phenotypes, covariates, prs_pheno_map):
    # Initialize an empty DataFrame to store residuals
    # all_df.set_index('sample', inplace=True)
    cov_prs_corrected_phenos = pd.DataFrame(index=all_df.index) # Index is the sample ID

    # Perform linear regression for each phenotype
    for pheno in tqdm(phenotypes):
        # Drop NaN values for the current phenotype
        combined_df = all_df[[pheno] + covariates + [prs_pheno_map[pheno]]].dropna()
        y = combined_df[pheno]
        X = combined_df.drop(columns=[pheno])
        X = sm.add_constant(X)  # Add a constant term for the intercept

        # Fit the model
        model = sm.OLS(y, X).fit()

        # Save residuals
        residuals = pd.Series(model.resid, index=combined_df.index, name=pheno)
        cov_prs_corrected_phenos = pd.concat([cov_prs_corrected_phenos, residuals], axis=1)

    # Reset the index for the resulting DataFrame
    cov_prs_corrected_phenos.reset_index(inplace=True)
    # cov_prs_corrected_phenos.columns = ['sample'] + [f"{pheno}_cov_prs_corrected" for pheno in phenotypes]
    return cov_prs_corrected_phenos


In [ ]:
def pheno_burden_spearman(assoc_df, gt_df, annotation):
    rank_corr_list = []
    for trait in assoc_df.phenotype.unique():
        gene_list = list(assoc_df.query("phenotype == @trait").gene.astype(str))
        pheno = trait.replace(" ", "_")
        for gene in gene_list:
            try:
                correlation = gt_df[[gene, pheno]].dropna().corr(method='spearman').iloc[0, 1]
                # Calculate the correlation only for non-zero values
                gis_mode = gt_df[gene].mode()[0]
                correlation_non_zero = gt_df[gt_df[gene]!=gis_mode][[gene, pheno]].dropna().corr(method='spearman').iloc[0, 1]
            except Exception as e:
                print(f"Cannot compute correlation for {annotation}, {pheno}, {gene}. Error: {e}")
                correlation = np.nan
                correlation_non_zero = np.nan
            
            rank_corr_list.append(
                pd.DataFrame({
                    'annotation': annotation,
                    'phenotype': trait,
                    'gene': gene,
                    'spearman_correlation': correlation,
                    'spearman_correlation_non_zero': correlation_non_zero,
                }, index=[0])
            )
    return pd.concat(rank_corr_list)

In [ ]:
def compute_correlations(config_path, zarr_burdens_path, associations_df_path, genes_to_keep = None):

    with open(config_path) as f:
        config = yaml.safe_load(f)

    anngeno_file = config.get('anngeno_file')
    annotation_list = config.get('rare_variant_annotations')
    phenotypes = config.get('phenotypes_for_association_testing')
    covs = config.get('covariates')
    prs_pheno_map_file = config.get('prs_pheno_map_file')
    prs_file = config.get('prs_file')

    cov_pheno_df = pd.read_parquet(f"{anngeno_file}/phenotypes.parquet", columns=['sample'] + covs + phenotypes).set_index('sample')
    prs_df = pd.read_parquet(prs_file)
    prs_df = prs_df[prs_df.index.isin(cov_pheno_df.index)]
    prs_pheno_map = pd.read_csv(prs_pheno_map_file)
    prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
    all_df = pd.concat([cov_pheno_df, prs_df], axis=1)
    pheno_corrected_df = cov_prs_correction(all_df, phenotypes, covs, prs_pheno_map)

    zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
    sample_list = zarr_group['samples'][:]
    gene_list = zarr_group["genes"][:]
    annotation_list = zarr_group["annotations"][:]

    assoc_df = pd.read_parquet(associations_df_path)
    if genes_to_keep is not None:
        assoc_df = assoc_df[assoc_df.gene.isin(genes_to_keep)]
    rho_df_sum_list = []
    rho_df_max_list = []
    for anno in tqdm(annotation_list):
        anno_idx = np.where(annotation_list == anno)[0][0]
        sum_burdens = zarr_group["sum_burdens"][:, :, anno_idx]
        gt_df_sum = pd.DataFrame(sum_burdens, index=sample_list, columns=gene_list).merge(pheno_corrected_df, left_index=True, right_on='sample')
        rho_df_sum_list.append(pheno_burden_spearman(assoc_df, gt_df_sum, anno))
        max_burdens = zarr_group["max_burdens"][:, :, anno_idx]
        gt_df_max = pd.DataFrame(max_burdens, index=sample_list, columns=gene_list).merge(pheno_corrected_df, left_index=True, right_on='sample')
        rho_df_max_list.append(pheno_burden_spearman(assoc_df, gt_df_max, anno))

    rho_df_sum = pd.concat(rho_df_sum_list)
    rho_df_sum['aggregation'] = 'sum'
    rho_df_max = pd.concat(rho_df_max_list)
    rho_df_max['aggregation'] = 'max'
    rho_df = pd.concat([rho_df_sum, rho_df_max])
    return rho_df

In [ ]:
config_path = './config.yaml'
zarr_burdens_path = '/s/project/deeprvat/ukb_gym/burdens/burdens_all_no_vars_na.zarr'
associations_df_path = '/s/project/deeprvat/ukb_gym/161k_plof_associations.pq' 

rho_df = compute_correlations(config_path, zarr_burdens_path, associations_df_path)
rho_df.to_parquet('/s/project/deeprvat/ukb_gym/data/spearman_rho/correlation_all_no_vars_na.parquet')
rho_df

In [ ]:
def pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_burdens_path):

    with open(config_path) as f:
        config = yaml.safe_load(f)

    anngeno_file = config.get('anngeno_file')
    # annotation_list = config.get('rare_variant_annotations')
    # phenotypes = config.get('phenotypes_for_association_testing')
    covs = config.get('covariates')
    prs_pheno_map_file = config.get('prs_pheno_map_file')
    prs_file = config.get('prs_file')

    cov_pheno_df = pd.read_parquet(f"{anngeno_file}/phenotypes.parquet", columns=['sample'] + covs + [phenotype]).set_index('sample')
    prs_df = pd.read_parquet(prs_file)
    prs_df = prs_df[prs_df.index.isin(cov_pheno_df.index)]
    prs_pheno_map = pd.read_csv(prs_pheno_map_file)
    prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
    all_df = pd.concat([cov_pheno_df, prs_df], axis=1)
    pheno_corrected_df = cov_prs_correction(all_df, [phenotype], covs, prs_pheno_map)

    zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
    sample_list = zarr_group['samples'][:]
    gene_list = zarr_group["genes"][:]
    annotation_list = zarr_group["annotations"][:]

    gene_idx = np.where(gene_list == gene_num)[0][0]
    anno_idx = np.where(annotation_list == annotation)[0][0]
    burden_type = burden_type.lower()
    if burden_type == "max":
        burdens = zarr_group["max_burdens"][:, gene_idx, anno_idx]
    elif burden_type == "sum":
        burdens = zarr_group["sum_burdens"][:, gene_idx, anno_idx]
    else:
        raise ValueError("burden_type must be either 'max' or 'sum'")

    burden_df = pd.DataFrame(burdens, index=sample_list, columns=[gene_num]).merge(pheno_corrected_df, left_index=True, right_on='sample')
    burden_df = burden_df.dropna()
    gis_mode = burden_df[gene_num].mode()[0]
    burden_df_non_zero = burden_df[burden_df[gene_num]!=gis_mode]
    correlation = burden_df[[gene_num, phenotype]].dropna().corr(method='spearman').iloc[0, 1]

    return burden_df, burden_df_non_zero, correlation


In [ ]:
phenotype = 'LDL_direct_statin_corrected'
gene_num = '9138'
annotation = 'model_22_max'
burden_type = 'sum'
plt_df, plt_df_nz, _ = pheno_gis_plot(phenotype, gene_num, annotation, burden_type, config_path, zarr_burdens_path)

plt_df_nz

In [ ]:
plt_df, plt_df_nz, c = pheno_gis_plot(phenotype, gene_num, "model_22_max", burden_type, config_path, zarr_burdens_path)

(
    ggplot(plt_df, aes(x=gene_num, y=phenotype)) +
    geom_point(alpha=0.5) +
    geom_smooth(method='lm', se=False) +
    labs(x='LDLR',
         y=phenotype) +
    theme_bw() +
    labs(title = f"AbSplice2 - {round(c, 4)}")

)

In [ ]:
plt_df, plt_df_nz, c = pheno_gis_plot(phenotype, gene_num, "pangolin_score", burden_type, config_path, zarr_burdens_path)

(
    ggplot(plt_df, aes(x=gene_num, y=phenotype)) +
    geom_point(alpha=0.5) +
    geom_smooth(method='lm', se=False) +
    labs(x='LDLR',
         y=phenotype) +
    theme_bw()+
    labs(title = f"Pangolin - {round(c, 4)}")
)

### Debugging

In [ ]:
rho_df.groupby('annotation')['spearman_correlation'].apply(lambda x: x.isna().sum()).sort_values()

In [ ]:
burden_file = "/s/project/deeprvat/ukb_gym/debugging/gene_burdens.zarr"
zarr_group = zarr.open_group(burden_file, mode="r")
sample_list = zarr_group['samples'][:]
gene_list = zarr_group["genes"][:]
annotation_list = zarr_group["annotations"][:]


In [ ]:
anno2test = 'Consequence_stop_lost'
anno_idx = np.where(annotation_list == anno2test)[0][0]
burdens = zarr_group["gene_burdens"][:, :, anno_idx]
g_df = pd.DataFrame(burdens, index=sample_list, columns=gene_list)
g_df

In [ ]:
g_df.sum().sort_values(ascending=False)

In [ ]:
g_df.sum().hist(bins=100)
plt.show()

## Make plots

In [ ]:
list(rank_corr_df["annotation"].unique())


In [ ]:
annotations_to_exclude = ['model_22_max','AbSplice_DNA_max', 'pangolin_score']

In [ ]:
rank_corr_df = pd.read_parquet('/s/project/deeprvat/ukb_gym/data/spearman_rho/correlation_all_no_vars_na.parquet')
rank_corr_df = rank_corr_df[~rank_corr_df['annotation'].isin(annotations_to_exclude)]
config_path = './config.yaml'  # Or wherever your config file is

with open(config_path) as f:
    config = yaml.safe_load(f)

rare_variant_annotations_dict = config.get('rare_variant_annotations')

# Create a mapping from annotation name to category
annotation_category_map = {}
if rare_variant_annotations_dict:
    for category_name, annotations in rare_variant_annotations_dict.items():
        for annotation in annotations:
            annotation_category_map[annotation] = category_name

# Add a 'category' column to rank_corr_df based on the mapping
rank_corr_df['category'] = rank_corr_df['annotation'].map(annotation_category_map)
rank_corr_df['category'] = pd.Categorical(rank_corr_df['category'], categories=['plof', 'missense', 'splicing', 'regulatory', 'misc'], ordered=True)
rank_corr_df

In [ ]:
rank_corr_df.query("phenotype == 'LDL direct statin corrected'")

In [ ]:
# rank_corr_df['abs_correlation_non_zero'] = np.abs(rank_corr_df['spearman_correlation_non_zero'])
# rank_corr_df['median_abs_correlation_non_zero'] = rank_corr_df.groupby('annotation')['abs_correlation_non_zero'].transform('median')
# rank_corr_df = rank_corr_df.sort_values('median_abs_correlation_non_zero', ascending=False)
# rank_corr_df['annotation'] = pd.Categorical(rank_corr_df['annotation'], categories=rank_corr_df['annotation'].unique(), ordered=True)

# (
#     ggplot(rank_corr_df.query("aggregation == 'sum'"), aes(x='annotation', y='abs_correlation_non_zero')) +
#     geom_boxplot() +
#     theme_bw() +
#     # scale_y_log10() +
#     # scale_y_sqrt() +
#     ylab('|rank correlation|') +
#     facet_wrap('~category', scales='free') +
#     theme(
#         axis_text_x=element_text(rotation=90, vjust=1),
#         figure_size=(12, 10),
#     )
# )

In [ ]:
rank_corr_df['abs_correlation'] = np.abs(rank_corr_df['spearman_correlation'])
rank_corr_df['median_abs_correlation'] = rank_corr_df.groupby('annotation')['abs_correlation'].transform('median')
rank_corr_df = rank_corr_df.sort_values('median_abs_correlation', ascending=False)
rank_corr_df['annotation'] = pd.Categorical(rank_corr_df['annotation'], categories=rank_corr_df['annotation'].unique(), ordered=True)

(
    ggplot(rank_corr_df.query("aggregation == 'max'"), aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    # scale_y_log10() +
    # scale_y_sqrt() +
    ylab('|rank correlation|') +
    facet_wrap('~category', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(12, 10),
    )
)

In [ ]:
rank_corr_df['abs_correlation'] = np.abs(rank_corr_df['spearman_correlation'])
rank_corr_df['median_abs_correlation'] = rank_corr_df.groupby('annotation')['abs_correlation'].transform('median')
rank_corr_df = rank_corr_df.sort_values('median_abs_correlation', ascending=False)
rank_corr_df['annotation'] = pd.Categorical(rank_corr_df['annotation'], categories=rank_corr_df['annotation'].unique(), ordered=True)

(
    ggplot(rank_corr_df.query("aggregation == 'sum'"), aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    # scale_y_log10() +
    # scale_y_sqrt() +
    ylab('|rank correlation|') +
    facet_wrap('~category', scales='free') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(12,10),
    )
)

In [ ]:
rank_corr_wide = rank_corr_df.query("aggregation == 'max'")\
    .pivot(index = ["phenotype", "gene"], columns = "annotation", values = "abs_correlation").reset_index()
rank_corr_wide

In [ ]:
comp1 = "influence_score_upstream"
comp2 = "GPN_MSA_score"


In [ ]:
rank_corr_wide = rank_corr_wide[["phenotype", "gene", comp1, comp2]]
rank_corr_wide

In [ ]:
rank_corr_wide.query("gene == '9138'")

In [ ]:
(ggplot(rank_corr_wide, aes(x=comp1, y=comp2)) + 
 geom_point() +
 geom_abline() +
 theme_bw() +
 theme(figure_size=(5, 5)) +
#  ylim(0, 0.032) + 
#  xlim(0, 0.032) +
 labs(title = "|rank correlation|"))

In [ ]:
rank_corr_wide = rank_corr_df.query("aggregation == 'max'").pivot(index=["phenotype", "gene"], columns="annotation", values="abs_correlation").reset_index()
rank_corr_wide

In [ ]:
rank_corr_wide["gene"].unique()

In [ ]:
rank_corr_df[['annotation', "phenotype", "gene"]]

In [ ]:
rank_corr_df[['annotation',"median_abs_correlation" ]].drop_duplicates()

In [ ]:
(
    ggplot(rank_corr_df, aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    scale_y_sqrt() +
    ylab('|rank correlation|') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(10, 10),
    )
)